# 15 — Weekly Forecast Analysis
## SunnyBest Retail Forecasting System

> **Purpose:** Read, explore and visualise the weekly unit forecasts generated by the model.  
> Run this notebook after `generate_weekly_forecast.py` to understand what the model is predicting.

| Section | What you see |
|---------|-------------|
| 1 | What weeks and models are in the forecast file |
| 2 | Forecast summary — total units per store |
| 3 | Forecast by category |
| 4 | Top 10 highest and lowest predicted products |
| 5 | Store × product heatmap |

---
## 0. Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sqlalchemy import create_engine
from urllib.parse import quote_plus

plt.rcParams["figure.figsize"]    = (14, 5)
plt.rcParams["axes.spines.top"]   = False
plt.rcParams["axes.spines.right"] = False
PALETTE = ["#4C72B0","#C44E52","#55A868","#DD8452","#8172B2","#64B5CD","#CCB974"]

FORECAST_PATH = "../data/outputs/weekly_forecasts.csv"

# DB connection — to enrich with product/store names
host     = "aws-1-eu-central-1.pooler.supabase.com"
port     = 5432
database = "postgres"
user     = "postgres.ogkdfmkybqtrsglcizzt"
password = quote_plus(os.getenv("SUPABASE_DB_PASSWORD", "Bonabosssfs01"))

engine = create_engine(
    f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}",
    pool_pre_ping=True, pool_recycle=300
)

print("Setup complete")

---
## 1. Load Forecasts & Enrich with Names

In [ ]:
# Load forecasts
df = pd.read_csv(FORECAST_PATH)
df["week_start"]     = pd.to_datetime(df["week_start"])
df["store_id"]       = df["store_id"].astype(str)
df["product_id"]     = pd.to_numeric(df["product_id"], errors="coerce").astype(int)
df["predicted_units"]= pd.to_numeric(df["predicted_units"], errors="coerce")

# Load dim tables for names
stores   = pd.read_sql("SELECT store_id::text, store_name, store_size, city, region FROM core.dim_stores", engine)
products = pd.read_sql("SELECT product_id, product_name, category, regular_price FROM core.dim_products", engine)
products["product_id"] = products["product_id"].astype(int)

# Enrich
df = df.merge(stores,   on="store_id",   how="left")
df = df.merge(products, on="product_id", how="left")

# Latest forecast week only
latest_week = df["week_start"].max()
fc = df[df["week_start"] == latest_week].copy()

print(f"Forecast week  : {latest_week.date()}")
print(f"Model used     : {fc['model_version'].iloc[0]}")
print(f"Predictions    : {len(fc):,}  ({fc['store_id'].nunique()} stores × {fc['product_id'].nunique()} products)")
print(f"Total predicted: {fc['predicted_units'].sum():,.0f} units")
print()
display(fc[["store_name","product_name","category","predicted_units","regular_price"]].head(15))

---
## 2. Forecast Summary by Store

> Which stores are expected to sell the most units this week?

In [ ]:
by_store = (fc.groupby(["store_name","store_size","city"])
              .agg(predicted_units=("predicted_units","sum"),
                   products       =("product_id","nunique"))
              .reset_index()
              .sort_values("predicted_units", ascending=False))

by_store["predicted_units"] = by_store["predicted_units"].round(1)
display(by_store)

fig, ax = plt.subplots(figsize=(12, 5))
colors = [PALETTE[i % len(PALETTE)] for i in range(len(by_store))]
ax.barh(by_store["store_name"][::-1], by_store["predicted_units"][::-1], color=colors[::-1])
ax.set_xlabel("Predicted Units")
ax.set_title(f"Forecast Week {latest_week.date()} — Total Predicted Units by Store")
for i, (_, row) in enumerate(by_store[::-1].iterrows()):
    ax.text(row["predicted_units"] + 5, i, f"{row['predicted_units']:,.0f}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

---
## 3. Forecast by Category

> Which product categories drive the most volume this week?

In [ ]:
by_cat = (fc.groupby("category")
            .agg(predicted_units=("predicted_units","sum"),
                 products       =("product_id","nunique"),
                 avg_price      =("regular_price","mean"))
            .reset_index()
            .sort_values("predicted_units", ascending=False))

by_cat["predicted_revenue"] = (by_cat["predicted_units"] * by_cat["avg_price"]).round(0)
by_cat["predicted_units"]   = by_cat["predicted_units"].round(1)
display(by_cat)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(by_cat["category"], by_cat["predicted_units"],
            color=PALETTE[:len(by_cat)])
axes[0].set_title("Predicted Units by Category")
axes[0].set_ylabel("Units")
axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(by_cat["category"], by_cat["predicted_revenue"] / 1e6,
            color=PALETTE[:len(by_cat)])
axes[1].set_title("Predicted Revenue by Category (₦M)")
axes[1].set_ylabel("Revenue (₦M)")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

---
## 4. Top 10 Highest & Lowest Predicted Products

> Highest — fast movers to ensure are well stocked.  
> Lowest — slow movers to watch for dead stock.

In [ ]:
by_product = (fc.groupby(["product_id","product_name","category","regular_price"])
                .agg(predicted_units=("predicted_units","sum"))
                .reset_index()
                .sort_values("predicted_units", ascending=False))

top10    = by_product.head(10)
bottom10 = by_product.tail(10).sort_values("predicted_units")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(top10["product_name"][::-1], top10["predicted_units"][::-1], color=PALETTE[2])
axes[0].set_title("Top 10 — Highest Predicted Units")
axes[0].set_xlabel("Predicted Units")

axes[1].barh(bottom10["product_name"], bottom10["predicted_units"], color=PALETTE[1])
axes[1].set_title("Bottom 10 — Lowest Predicted Units")
axes[1].set_xlabel("Predicted Units")

plt.tight_layout()
plt.show()

print("Top 10:")
display(top10[["product_name","category","regular_price","predicted_units"]].reset_index(drop=True))
print("\nBottom 10:")
display(bottom10[["product_name","category","regular_price","predicted_units"]].reset_index(drop=True))

---
## 5. Store × Category Heatmap

> Which store-category combinations drive the most volume?  
> Useful for deciding where to focus replenishment effort this week.

In [ ]:
pivot = (fc.groupby(["store_name","category"])["predicted_units"]
           .sum()
           .round(1)
           .unstack(fill_value=0))

fig, ax = plt.subplots(figsize=(16, 6))
im = ax.imshow(pivot.values, cmap="YlGn", aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=30, ha="right", fontsize=9)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=9)
ax.set_title(f"Predicted Units — Store × Category  (week of {latest_week.date()})")
plt.colorbar(im, ax=ax, label="Predicted units")

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f"{pivot.values[i,j]:.0f}",
                ha="center", va="center", fontsize=7, color="black")

plt.tight_layout()
plt.show()

---
## 6. All Forecast Weeks (History)

> If you've run the forecast script multiple times, this shows all weeks side by side.

In [ ]:
all_weeks = (df.groupby(["week_start","model_version"])
               .agg(total_predicted=("predicted_units","sum"),
                    rows=("product_id","count"))
               .reset_index())

all_weeks["total_predicted"] = all_weeks["total_predicted"].round(0)
display(all_weeks)

if len(all_weeks) > 1:
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.bar(all_weeks["week_start"].dt.strftime("%Y-%m-%d"),
           all_weeks["total_predicted"], color=PALETTE[0])
    ax.set_title("Total Predicted Units Across All Forecast Weeks")
    ax.set_ylabel("Predicted Units")
    ax.set_xlabel("Forecast Week")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()
else:
    print("Only one forecast week so far — run the forecast script again next Saturday to build history.")